### Tools

Models can request to call tools that perform tasks such as fetching data from DB, searching web or running code. Tools are pairings of:

1. A Schema, including name of tool, description, and/or argument definition(often JSON Schema)
2. A function or coroutine to execute

In [4]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.8-27b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='The short answer is that parrots don’t actually "talk" in the human sense. They are **mimicking** sounds they hear, primarily to communicate with their flock and, in the case of pet parrots, to communicate with their human family.\n\nHere’s a deeper breakdown of why and how they do it:\n\n### 1. Social Bonding\nIn the wild, parrots are highly social birds. They live in flocks and use a variety of calls to:\n- Maintain contact with flock members.\n- Alert others to danger.\n- Establish territory.\n- Strengthen social bonds.\n\nWhen you bring a parrot into a home, it treats you as part of its flock. Learning your voice, sounds, and "words" is a way to bond with you and fit in.\n\n### 2. Vocal Anatomy\nParrots have a specialized vocal organ called the **syrinx** (located at the base of the trachea, not in the throat like humans). This allows them to produce a wide range of sounds, including:\n- Complex whistles.\n- Songs.\n- Mimicry of human speech, other animals, or ho

In [5]:
from langchain.tools import tool 

@tool
def get_weather(location:str) ->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [6]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'tool_calls': [{'id': 'kzr658c9x', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 278, 'total_tokens': 304, 'completion_time': 0.067333089, 'completion_tokens_details': None, 'prompt_time': 0.019566524, 'prompt_tokens_details': None, 'queue_time': 0.049994194, 'total_time': 0.086899613}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a14d5e-bf41-76e0-a528-91c865cc27bb-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'kzr658c9x', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 278, 'output_tokens': 26, 'total_tokens': 304}
Tool: get_weather
Args: {'location': 'Boston'}


In [8]:
tool_call

{'name': 'get_weather',
 'args': {'location': 'Boston'},
 'id': 'kzr658c9x',
 'type': 'tool_call'}

### Tool Execution Loops

In [9]:
# Step 1: Model generates tool calls
messages = [{"role":"user","content":"What's the weather like in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    #Execute tool with generated args
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

#Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

# "The current weather in Boston is 72F and sunny"

It's sunny in Boston today! ☀️


In [10]:
messages

[{'role': 'user', 'content': "What's the weather like in Boston?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'w27wyygqm', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 278, 'total_tokens': 304, 'completion_time': 0.068197736, 'completion_tokens_details': None, 'prompt_time': 0.018881845, 'prompt_tokens_details': None, 'queue_time': 0.049582153, 'total_time': 0.087079581}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_a1293f40b5', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a14d65-2486-7b33-8d82-8d300e947310-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'w27wyygqm', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 278, 'output_tokens': 26, 'total_tokens': 304}),
 ToolMessage(content="It's s

In [11]:
ai_msg

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'w27wyygqm', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 278, 'total_tokens': 304, 'completion_time': 0.068197736, 'completion_tokens_details': None, 'prompt_time': 0.018881845, 'prompt_tokens_details': None, 'queue_time': 0.049582153, 'total_time': 0.087079581}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_a1293f40b5', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a14d65-2486-7b33-8d82-8d300e947310-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'w27wyygqm', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 278, 'output_tokens': 26, 'total_tokens': 304})